In [3]:
#  SANITY CHECK
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import gc
import os
import math

from unet3d_2 import UNet3D
from luna16_dataset_and_dataloader import create_patch_dataloaders

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_DIR         = "/home/jovyan/vol.2/unet_patches"
BATCH_SIZE        = 4
NUM_WORKERS       = 2
POSITIVE_FRACTION = 0.75
INIT_FEATURES     = 48
DROPOUT           = 0.2
POS_WEIGHT        = 20.0
USE_AMP           = True


class ChannelAwareFocalLoss(nn.Module):
    def __init__(self, base_pos_weight=10.0, alpha=0.75, gamma=2.0):
        super().__init__()
        self.base_pos_weight = base_pos_weight
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets, channel_importances):
        all_imp  = torch.stack([imp.mean() for imp in channel_importances])
        mean_imp = all_imp.mean()
        dyn_pw   = self.base_pos_weight * (1.0 + mean_imp)
        pw_tensor = torch.tensor([dyn_pw.item()], device=inputs.device)
        bce = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=pw_tensor, reduction='none')
        pt  = torch.exp(-bce.detach())
        loss = torch.mean(self.alpha * (1 - pt) ** self.gamma * bce)
        return loss, dyn_pw.item()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, predict, target):
        predict = torch.sigmoid(predict).view(-1)
        target  = target.view(-1)
        intersection = (predict * target).sum()
        return 1 - (2. * intersection + self.smooth) / (predict.sum() + target.sum() + self.smooth)

_ca_focal = ChannelAwareFocalLoss(base_pos_weight=POS_WEIGHT)
_dice     = DiceLoss()

def sanity_combined_loss(pred, target, importances):
    pw_static = torch.tensor([POS_WEIGHT], device=pred.device)
    bce       = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pw_static)
    focal, dpw = _ca_focal(pred, target, importances)
    dice       = _dice(pred, target)
    return 0.4 * bce + 0.35 * focal + 0.25 * dice, dpw


def calculate_batch_metrics(predictions, targets, threshold=0.5):
    with torch.no_grad():
        binary = (torch.sigmoid(predictions) > threshold).float().view(-1)
        tgt    = targets.view(-1)
        TP = ((binary == 1) & (tgt == 1)).sum().float()
        FP = ((binary == 1) & (tgt == 0)).sum().float()
        FN = ((binary == 0) & (tgt == 1)).sum().float()
        TN = ((binary == 0) & (tgt == 0)).sum().float()

        recall      = TP / (TP + FN + 1e-8)
        precision   = TP / (TP + FP + 1e-8)
        dice        = (2 * TP) / (2 * TP + FP + FN + 1e-8)
        specificity = TN / (TN + FP + 1e-8)
        npv         = TN / (TN + FN + 1e-8)
        mcc_num     = TP * TN - FP * FN
        mcc_den     = ((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)).sqrt()
        mcc         = mcc_num / (mcc_den + 1e-8)

    return {
        'TP': TP.item(), 'FP': FP.item(), 'FN': FN.item(), 'TN': TN.item(),
        'recall':      recall.item(),
        'precision':   precision.item(),
        'dice':        dice.item(),
        'specificity': specificity.item(),
        'npv':         npv.item(),
        'mcc':         mcc.item(),
    }


print("Loading data for sanity check...")
train_loader, _, _ = create_patch_dataloaders(
    patch_dir=PATCH_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    positive_fraction=POSITIVE_FRACTION,
)


sanity_model = UNet3D(
    input_channels=1,
    output_channels=1,
    init_features=INIT_FEATURES,
    dropout=DROPOUT,
    checkpointing=False,
).to(device)

n_params = sum(p.numel() for p in sanity_model.parameters() if p.requires_grad)
print(f"Model params: {n_params/1e6:.2f}M  |  device: {device}\n")


N_EPOCHS  = 5
N_BATCHES = 100

checks = {
    'forward_pass':      False,
    'loss_decreasing':   False,
    'importance_moving': False,
    'recall_nonzero':    False,
    'no_oom':            False,
}

opt = optim.AdamW(sanity_model.parameters(), lr=5e-5)
sc  = GradScaler('cuda', enabled=USE_AMP)

epoch_losses  = []
epoch_recalls = []

imp_before = torch.cat([
    imp.detach().cpu() for imp in sanity_model.get_all_channel_importances()
])

print("=" * 65)
print(f"Running {N_EPOCHS} epochs × {N_BATCHES} batches...")
print(f"Metrics: recall, precision, dice, specificity, NPV, MCC")
print("=" * 65)

try:
    for epoch in range(N_EPOCHS):
        sanity_model.train()
        b_losses = []

        b_TP = b_FP = b_FN = b_TN = 0.0

        for step, batch in enumerate(train_loader):
            if step >= N_BATCHES:
                break

            scans = batch['scan'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)

            opt.zero_grad()
            with autocast('cuda', enabled=USE_AMP):
                outputs     = sanity_model(scans)
                importances = sanity_model.get_all_channel_importances()
                loss, dpw   = sanity_combined_loss(outputs, masks, importances)

            if torch.isnan(loss):
                raise ValueError("Loss is NaN - Check Scaling/Normalization")

            if epoch == 0 and step == 0:
                checks['forward_pass'] = True
                print(f"  [OK] Forward pass  — output shape: {outputs.shape}")
                print(f"       Initial dynamic pos_weight: {dpw:.2f}")

            sc.scale(loss).backward()
            sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(sanity_model.parameters(), 1.0)
            sc.step(opt)
            sc.update()

            b_losses.append(loss.item())
            m = calculate_batch_metrics(outputs, masks)
            b_TP += m['TP']; b_FP += m['FP']
            b_FN += m['FN']; b_TN += m['TN']

        ep_loss        = sum(b_losses) / len(b_losses)
        ep_recall      = b_TP / (b_TP + b_FN + 1e-8)
        ep_precision   = b_TP / (b_TP + b_FP + 1e-8)
        ep_dice        = (2 * b_TP) / (2 * b_TP + b_FP + b_FN + 1e-8)
        ep_specificity = b_TN / (b_TN + b_FP + 1e-8)
        ep_npv         = b_TN / (b_TN + b_FN + 1e-8)
        ep_mcc_num     = b_TP * b_TN - b_FP * b_FN
        ep_mcc_den     = ((b_TP + b_FP) * (b_TP + b_FN) * (b_TN + b_FP) * (b_TN + b_FN)) ** 0.5
        ep_mcc         = ep_mcc_num / (ep_mcc_den + 1e-8)

        epoch_losses.append(ep_loss)
        epoch_recalls.append(ep_recall)

        imp_means = [f"{i.mean().item():.4f}" for i in sanity_model.get_all_channel_importances()]

        print(f"\n  Epoch {epoch+1}/{N_EPOCHS}")
        print(f"    {'Metric':<14} {'Value':>10}")
        print(f"    {'Loss':<14} {ep_loss:>10.4f}")
        print(f"    {'Recall':<14} {ep_recall:>10.4f}")
        print(f"    {'Precision':<14} {ep_precision:>10.4f}")
        print(f"    {'Dice':<14} {ep_dice:>10.4f}")
        print(f"    {'Specificity':<14} {ep_specificity:>10.4f}")
        print(f"    {'NPV':<14} {ep_npv:>10.4f}")
        print(f"    {'MCC':<14} {ep_mcc:>10.4f}")
        print(f"    imp_means: [{' | '.join(imp_means)}]")

    checks['no_oom'] = True

    if epoch_losses[-1] < epoch_losses[0]:
        checks['loss_decreasing'] = True

    imp_after = torch.cat([
        imp.detach().cpu() for imp in sanity_model.get_all_channel_importances()
    ])
    delta = (imp_after - imp_before).abs().mean().item()
    if delta > 1e-5:
        checks['importance_moving'] = True

    if epoch_recalls[-1] > 0.001:
        checks['recall_nonzero'] = True

except torch.cuda.OutOfMemoryError:
    print("\n  [FAIL] CUDA OOM")
    checks['no_oom'] = False
except Exception as e:
    print(f"\n  [ERROR] {str(e)}")
    all_passed = False


print("\n" + "=" * 65)
print("SANITY CHECK RESULTS")
print("=" * 65)

messages = {
    'forward_pass':      ("Forward pass runs cleanly",
                          "Shape error — check INIT_FEATURES or input dims"),
    'loss_decreasing':   (f"Loss decreasing  {epoch_losses[0]:.4f} → {epoch_losses[-1]:.4f}",
                          f"Loss NOT decreasing  {epoch_losses[0] if epoch_losses else 0:.4f} → {epoch_losses[-1] if epoch_losses else 0:.4f}"),
    'importance_moving': (f"channel_importance learning  Δ={delta:.6f}",
                          f"channel_importance FROZEN  Δ={delta if 'delta' in locals() else 0:.6f}"),
    'recall_nonzero':    (f"Recall non-zero: {epoch_recalls[-1]:.4f}",
                          f"Recall near zero: {epoch_recalls[-1] if epoch_recalls else 0:.4f}"),
    'no_oom':            ("No OOM error",
                          "OOM detected"),
}

all_passed = True
for key, ok in checks.items():
    tag = "PASS" if ok else "FAIL"
    msg = messages[key][0] if ok else messages[key][1]
    print(f"  [{tag}]  {msg}")
    if not ok:
        all_passed = False

print()
if all_passed:
    print("  ✓ ALL CHECKS PASSED")
    print("  → Safe to run Cell 2 (full training)")
else:
    print("  ✗ FIX FAILURES ABOVE")

print("=" * 65)

del sanity_model
gc.collect()
torch.cuda.empty_cache()

Loading data for sanity check...
  TRAIN: 49518 patches  | 30168 positive  | 19350 negative
  VAL  : 6880 patches  | 4392 positive  | 2488 negative
  TEST : 14640 patches  | 8712 positive  | 5928 negative

  Loaders | batch=4 | workers=2 | pos_fraction=0.75
  train=12380 batches | val=1720 | test=3660
Model params: 51.78M  |  device: cuda

Running 5 epochs × 100 batches...
Metrics: recall, precision, dice, specificity, NPV, MCC
  [OK] Forward pass  — output shape: torch.Size([4, 1, 96, 96, 96])
       Initial dynamic pos_weight: 30.00

  Epoch 1/5
    Metric              Value
    Loss               0.3387
    Recall             0.2609
    Precision          0.0547
    Dice               0.0904
    Specificity        0.9928
    NPV                0.9988
    MCC                0.1165
    imp_means: [0.5000 | 0.5000 | 0.5000 | 0.5000]

  Epoch 2/5
    Metric              Value
    Loss               0.2758
    Recall             0.5241
    Precision          0.1478
    Dice              

In [ ]:
import os
print(os.getcwd())

In [ ]:
import subprocess
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import os
import gc

from unet3d_2 import UNet3D
from luna16_dataset_and_dataloader import create_patch_dataloaders

try:
    __import__("codecarbon")
    print("codecarbon is already installed.")
except ImportError:
    print("codecarbon not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "codecarbon"])

from codecarbon import EmissionsTracker

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"



#LOSS FUNCTIONS

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        bce = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt  = torch.exp(-bce)
        return torch.mean(self.alpha * (1 - pt) ** self.gamma * bce)

class ChannelAwareFocalLoss(nn.Module):
    """
    Scales pos_weight dynamically via learned channel_importance.
    """
    def __init__(self, base_pos_weight: float = 10.0, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.base_pos_weight = base_pos_weight
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets, channel_importances):
        all_imp    = torch.stack([imp.mean() for imp in channel_importances])
        mean_imp   = all_imp.mean()
        dyn_pw     = self.base_pos_weight * (1.0 + mean_imp)
        pos_weight = torch.tensor([dyn_pw.item()], device=inputs.device)
        bce        = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')(inputs, targets)
        pt         = torch.exp(-bce)
        return torch.mean(self.alpha * (1 - pt) ** self.gamma * bce), dyn_pw.item()

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, predict, target):
        predict      = torch.sigmoid(predict).view(-1)
        target       = target.view(-1)
        intersection = (predict * target).sum()
        return 1 - (2. * intersection + self.smooth) / (
            predict.sum() + target.sum() + self.smooth)

_ca_focal = None
_dice     = DiceLoss()

def combined_loss(pred, target, channel_importances, pos_weight_val, label_smoothing=0.0):
    """
    0.4 * BCE + 0.35 * ChannelAwareFocal + 0.25 * Dice
    Dice always uses hard targets (smoothing distorts overlap geometry).
    """
    global _ca_focal
    if _ca_focal is None:
        _ca_focal = ChannelAwareFocalLoss(base_pos_weight=pos_weight_val)

    pos_weight_t = torch.tensor([pos_weight_val], device=pred.device)

    if label_smoothing > 0:
        smooth_target = target * (1 - label_smoothing) + 0.5 * label_smoothing
        bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)(pred, smooth_target)
    else:
        bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)(pred, target)

    focal, dyn_pw = _ca_focal(pred, target, channel_importances)
    dice          = _dice(pred, target)

    return 0.4 * bce + 0.35 * focal + 0.25 * dice, dyn_pw

#  METRICS (added MCC, specificity, NPV)

def calculate_batch_metrics(predictions, targets, threshold=0.5):
    binary = (torch.sigmoid(predictions) > threshold).float().view(-1)
    tgt    = targets.view(-1)
    TP = ((binary == 1) & (tgt == 1)).sum().float()
    FP = ((binary == 1) & (tgt == 0)).sum().float()
    FN = ((binary == 0) & (tgt == 1)).sum().float()
    TN = ((binary == 0) & (tgt == 0)).sum().float()

    recall      = TP / (TP + FN + 1e-8)
    precision   = TP / (TP + FP + 1e-8)
    dice        = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    npv         = TN / (TN + FN + 1e-8)
    mcc_num     = TP * TN - FP * FN
    mcc_den     = ((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)).sqrt()
    mcc         = mcc_num / (mcc_den + 1e-8)

    return {
        'TP': TP.item(), 'FP': FP.item(), 'FN': FN.item(), 'TN': TN.item(),
        'recall':      recall.item(),
        'precision':   precision.item(),
        'dice':        dice.item(),
        'specificity': specificity.item(),
        'npv':         npv.item(),
        'mcc':         mcc.item(),
    }

#  CONFIGURATION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PATCH_DIR         = "/home/jovyan/vol.2/unet_patches"
BATCH_SIZE        = 4
POSITIVE_FRACTION = 0.75
NUM_WORKERS       = 2
INIT_FEATURES     = 48
DROPOUT           = 0.2
INITIAL_LR        = 1e-4
WEIGHT_DECAY      = 1e-5
BETAS             = (0.9, 0.999)
EPS               = 1e-8
T_0               = 15
T_MULT            = 2
ETA_MIN           = 1e-6
POS_WEIGHT        = 20.0
LABEL_SMOOTHING   = 0.0
ACCUM_STEPS       = 4
USE_AMP           = True
GRAD_CLIP         = 1.0
NUM_EPOCHS        = 80
EARLY_STOPPING_PATIENCE = 30

os.makedirs('./checkpoints/all_epochs', exist_ok=True)

print("=" * 65)
print("U-NET TRAINING  v8")
print("=" * 65)
print(f"  patch_dir        : {PATCH_DIR}")
print(f"  batch_size       : {BATCH_SIZE}  (effective {BATCH_SIZE*ACCUM_STEPS} with accum)")
print(f"  optimizer        : AdamW  lr={INITIAL_LR}  wd={WEIGHT_DECAY}")
print(f"  scheduler        : CosineWarmRestarts  T0={T_0}  Tmult={T_MULT}")
print(f"  pos_weight       : {POS_WEIGHT}  (+ dynamic ChannelAware scaling)")
print(f"  focal alpha      : 0.75  |  loss: 0.4*BCE + 0.35*Focal + 0.25*Dice")
print(f"  dropout          : {DROPOUT}  |  AMP: {USE_AMP}")
print(f"  epochs           : {NUM_EPOCHS}  |  patience: {EARLY_STOPPING_PATIENCE}")
print(f"  metrics          : recall, precision, dice, specificity, NPV, MCC")
print("=" * 65)

#DATA
print("\nLoading datasets...")
train_loader, val_loader, test_loader = create_patch_dataloaders(
    patch_dir=PATCH_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    positive_fraction=POSITIVE_FRACTION,
)

# MODEL + OPTIMIZER + SCHEDULER
model = UNet3D(
    input_channels=1,
    output_channels=1,
    init_features=INIT_FEATURES,
    dropout=DROPOUT,
    checkpointing=False,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel params: {n_params/1e6:.2f}M  |  device: {device}")

optimizer = optim.AdamW(
    model.parameters(), lr=INITIAL_LR, betas=BETAS, eps=EPS, weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=T_0, T_mult=T_MULT, eta_min=ETA_MIN,
)
scaler = GradScaler(enabled=USE_AMP)

#  RESUME FROM CHECKPOINT
RESUME_FROM = './checkpoints/last_checkpoint.pth'
start_epoch = 0

if os.path.exists(RESUME_FROM):
    print(f"\nResuming from checkpoint: {RESUME_FROM}")
    checkpoint = torch.load(RESUME_FROM, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    start_epoch        = checkpoint['epoch']
    best_val_recall    = checkpoint['best_val_recall']
    best_balanced_dice = checkpoint['best_balanced_dice']

    train_losses      = checkpoint['train_losses']
    val_losses        = checkpoint['val_losses']
    train_recalls     = checkpoint['train_recalls']
    val_recalls       = checkpoint['val_recalls']
    train_precisions  = checkpoint['train_precisions']
    val_precisions    = checkpoint['val_precisions']
    train_dices       = checkpoint['train_dices']
    val_dices         = checkpoint['val_dices']
    learning_rates    = checkpoint['learning_rates']

    epochs_no_improve = 0
    for r in reversed(val_recalls):
        if r < best_val_recall:
            epochs_no_improve += 1
        else:
            break

    print(f"  Resumed at epoch {start_epoch + 1}")
    print(f"  best_recall={best_val_recall:.4f}  best_dice={best_balanced_dice:.4f}")
    print(f"  No-improve streak: {epochs_no_improve}")
else:
    print("\nNo checkpoint found — starting fresh.")
    best_val_recall    = 0.0
    best_balanced_dice = 0.0
    epochs_no_improve  = 0
    train_losses, val_losses         = [], []
    train_recalls, val_recalls       = [], []
    train_precisions, val_precisions = [], []
    train_dices, val_dices           = [], []
    learning_rates                   = []

print("\nSTARTING TRAINING\n")


#CODECARBON tracker
tracker = EmissionsTracker(
    project_name="unet3d_luna16",
    log_level="error",
    save_to_file=False,
)
tracker.start()
training_start_time = time.time()


#TRAINING LOOP
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    train_TP = train_FP = train_FN = train_TN = 0.0
    epoch_dyn_pw = []

    current_lr = optimizer.param_groups[0]['lr']
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train] lr={current_lr:.2e}")
    optimizer.zero_grad()

    for step, batch in enumerate(pbar):
        scans = batch['scan'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            outputs      = model(scans)
            importances  = model.get_all_channel_importances()
            loss, dyn_pw = combined_loss(outputs, masks, importances, POS_WEIGHT, LABEL_SMOOTHING)
            loss         = loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_dyn_pw.append(dyn_pw)

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUM_STEPS

        with torch.no_grad():
            m = calculate_batch_metrics(outputs, masks)
            train_TP += m['TP'];  train_FP += m['FP']
            train_FN += m['FN'];  train_TN += m['TN']

        pbar.set_postfix({'loss': f"{loss.item() * ACCUM_STEPS:.4f}"})

    avg_train_loss      = train_loss / len(train_loader)
    train_recall        = train_TP / (train_TP + train_FN + 1e-8)
    train_precision     = train_TP / (train_TP + train_FP + 1e-8)
    train_dice          = (2 * train_TP) / (2 * train_TP + train_FP + train_FN + 1e-8)
    train_specificity   = train_TN / (train_TN + train_FP + 1e-8)
    train_npv           = train_TN / (train_TN + train_FN + 1e-8)
    train_mcc_num       = train_TP * train_TN - train_FP * train_FN
    train_mcc_den       = ((train_TP + train_FP) * (train_TP + train_FN) *
                           (train_TN + train_FP) * (train_TN + train_FN)) ** 0.5
    train_mcc           = train_mcc_num / (train_mcc_den + 1e-8)
    avg_dyn_pw          = sum(epoch_dyn_pw) / len(epoch_dyn_pw)

    model.eval()
    val_loss = 0.0
    val_TP = val_FP = val_FN = val_TN = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]  "):
            scans   = batch['scan'].to(device, non_blocking=True)
            masks   = batch['mask'].to(device, non_blocking=True)
            with autocast(enabled=USE_AMP):
                outputs     = model(scans)
                importances = model.get_all_channel_importances()
                v_loss, _   = combined_loss(outputs, masks, importances, POS_WEIGHT, LABEL_SMOOTHING)
                val_loss   += v_loss.item()
            m = calculate_batch_metrics(outputs, masks)
            val_TP += m['TP'];  val_FP += m['FP']
            val_FN += m['FN'];  val_TN += m['TN']

    avg_val_loss    = val_loss / len(val_loader)
    val_recall      = val_TP / (val_TP + val_FN + 1e-8)
    val_precision   = val_TP / (val_TP + val_FP + 1e-8)
    val_dice        = (2 * val_TP) / (2 * val_TP + val_FP + val_FN + 1e-8)
    val_specificity = val_TN / (val_TN + val_FP + 1e-8)
    val_npv         = val_TN / (val_TN + val_FN + 1e-8)
    val_mcc_num     = val_TP * val_TN - val_FP * val_FN
    val_mcc_den     = ((val_TP + val_FP) * (val_TP + val_FN) *
                       (val_TN + val_FP) * (val_TN + val_FN)) ** 0.5
    val_mcc         = val_mcc_num / (val_mcc_den + 1e-8)


    scheduler.step(epoch + 1)
    new_lr = optimizer.param_groups[0]['lr']


    train_losses.append(avg_train_loss);       val_losses.append(avg_val_loss)
    train_recalls.append(train_recall);        val_recalls.append(val_recall)
    train_precisions.append(train_precision);  val_precisions.append(val_precision)
    train_dices.append(train_dice);            val_dices.append(val_dice)
    learning_rates.append(current_lr)

    #SUMMARY
    imp_tensors = model.get_all_channel_importances()
    imp_means   = [f"{i.mean().item():.3f}" for i in imp_tensors]
    imp_maxes   = [f"{i.max().item():.3f}"  for i in imp_tensors]

    print(f"\nEPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"  {'Metric':<14} {'Train':>10} {'Val':>10} {'Δ':>10}")
    print(f"  {'Loss':<14} {avg_train_loss:>10.4f} {avg_val_loss:>10.4f} {avg_val_loss-avg_train_loss:>+10.4f}")
    print(f"  {'Recall':<14} {train_recall:>10.4f} {val_recall:>10.4f} {val_recall-train_recall:>+10.4f}")
    print(f"  {'Precision':<14} {train_precision:>10.4f} {val_precision:>10.4f} {val_precision-train_precision:>+10.4f}")
    print(f"  {'Dice':<14} {train_dice:>10.4f} {val_dice:>10.4f} {val_dice-train_dice:>+10.4f}")
    print(f"  {'Specificity':<14} {train_specificity:>10.4f} {val_specificity:>10.4f} {val_specificity-train_specificity:>+10.4f}")
    print(f"  {'NPV':<14} {train_npv:>10.4f} {val_npv:>10.4f} {val_npv-train_npv:>+10.4f}")
    print(f"  {'MCC':<14} {train_mcc:>10.4f} {val_mcc:>10.4f} {val_mcc-train_mcc:>+10.4f}")
    print(f"  LR: {current_lr:.2e} → {new_lr:.2e}")
    print(f"  Dynamic pos_weight (avg): {avg_dyn_pw:.2f}")
    print(f"  Importance mean  (dec4→dec1): {' | '.join(imp_means)}")
    print(f"  Importance max   (dec4→dec1): {' | '.join(imp_maxes)}")

    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        resv  = torch.cuda.memory_reserved() / 1e9
        print(f"  GPU: {alloc:.1f}GB alloc / {resv:.1f}GB reserved")
        if resv > 20:
            print(f"  WARNING: High GPU memory ({resv:.1f}GB)")
        torch.cuda.reset_peak_memory_stats()

    #CHECKPOINT
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'train_losses': train_losses,         'val_losses': val_losses,
        'train_recalls': train_recalls,       'val_recalls': val_recalls,
        'train_precisions': train_precisions, 'val_precisions': val_precisions,
        'train_dices': train_dices,           'val_dices': val_dices,
        'learning_rates': learning_rates,
        'best_val_recall': best_val_recall,
        'best_balanced_dice': best_balanced_dice,
        'config': {
            'version': 'v8',
            'batch_size': BATCH_SIZE, 'accum_steps': ACCUM_STEPS,
            'optimizer': 'AdamW', 'initial_lr': INITIAL_LR,
            'weight_decay': WEIGHT_DECAY, 'betas': BETAS,
            'scheduler': 'CosineAnnealingWarmRestarts',
            'T_0': T_0, 'T_mult': T_MULT, 'eta_min': ETA_MIN,
            'pos_weight': POS_WEIGHT, 'focal_alpha': 0.75,
            'loss_weights': '0.4*BCE + 0.35*ChannelAwareFocal + 0.25*Dice',
            'dropout': DROPOUT, 'patch_size': (96, 96, 96),
            'amp': USE_AMP, 'device': str(device),
        },
    }

    torch.save(checkpoint, f'./checkpoints/all_epochs/epoch_{epoch+1:02d}.pth')
    torch.save(checkpoint, './checkpoints/last_checkpoint.pth')

    #EARLY STOPPING
    if val_recall > best_val_recall:
        best_val_recall = val_recall
        epochs_no_improve = 0
        torch.save(checkpoint, './checkpoints/best_recall.pth')
        print(f"  ✓ Best recall: {val_recall:.4f}  (dice={val_dice:.4f}  precision={val_precision:.4f}  MCC={val_mcc:.4f})")
        if val_recall >= 0.95:
            torch.save(checkpoint, './checkpoints/best_balanced.pth')
            print(f"  ★ TARGET HIT  recall={val_recall:.4f}  precision={val_precision:.4f}  dice={val_dice:.4f}  MCC={val_mcc:.4f}")
    else:
        epochs_no_improve += 1

    if val_dice > best_balanced_dice:
        best_balanced_dice = val_dice

    print(f"  No-improve streak: {epochs_no_improve}/{EARLY_STOPPING_PATIENCE}")

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f"\nEARLY STOPPING after {epoch+1} epochs.")
        break


#CODECARBON—stop tracker
emissions        = tracker.stop() * 1000         
energy_wh        = tracker._total_energy.kWh * 1000 
execution_time   = time.time() - training_start_time

print(f"\n~ : ~ consumption measured through CodeCarbon ~ : ~")
print(f"Python energy:         {energy_wh:.6f} Wh")
print(f"Python emissions:      {emissions:.6f} CO₂eq grams")
print(f"Python execution time: {execution_time:.2f} seconds  ({execution_time/3600:.2f} hours)")




#SUMMARY
print("\n" + "=" * 65)
print("TRAINING COMPLETE")
print(f"  Best recall        : {best_val_recall:.4f}")
print(f"  Best balanced dice : {best_balanced_dice:.4f}")
if best_balanced_dice > 0 and best_balanced_dice in val_dices:
    best_idx = val_dices.index(best_balanced_dice)
    print(f"  Best epoch         : {best_idx + 1}")
    print(f"    recall    = {val_recalls[best_idx]:.4f}")
    print(f"    precision = {val_precisions[best_idx]:.4f}")
    print(f"    dice      = {val_dices[best_idx]:.4f}")
print(f"\nCheckpoints in: ./checkpoints/")
print(f"  best_recall.pth     – highest val recall")
print(f"  best_balanced.pth   – recall ≥ 0.95 + best Dice")
print(f"  last_checkpoint.pth – resume from here after restart")
print("=" * 65)

codecarbon not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 28.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [codecarbon]3 [codecarbon]
U-NET TRAINING  v8
  patch_dir        : /home/jovyan/vol.2/unet_patches
  batch_size       : 4  (effective 16 with accum)
  optimizer        : AdamW  lr=0.0001  wd=1e-05
  scheduler        : CosineWarmRestarts  T0=15  Tmult=2
  pos_weight       : 20.0  (+ dynamic ChannelAware scaling)
  focal alpha      : 0.75  |  loss: 0.4*BCE + 0.35*Focal + 0.25*Dice
  dropout          : 0.2  |  AMP: True
  epochs           : 80  |  patience: 30
  metrics          : recall, precision, dice, specificity, NPV, MCC

Loading datasets...
  TRAIN: 49518 patches  | 30168 positive  | 19350 negative
  VAL  : 6880 patches  | 4392 positive  | 2488 negative
  TEST : 14640 patches  | 8712 positive  | 5928 negative

  Loaders | batch=4 | worke

/tmp/ipykernel_52/3408498194.py:223: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
[codecarbon WARNING @ 20:29:47] Multiple instances of codecarbon are allowed to run at the same time.



Model params: 51.78M  |  device: cuda

No checkpoint found — starting fresh.

STARTING TRAINING



Epoch 1/80 [Train] lr=1.00e-04:   0%|          | 0/12380 [00:00<?, ?it/s]/tmp/ipykernel_52/3408498194.py:315: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 1/80 [Val]  :   0%|          | 0/1720 [00:00<?, ?it/s]/tmp/ipykernel_52/3408498194.py:361: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 1/80 [Val]  : 100%|██████████| 1720/1720 [06:34<00:00,  4.36it/s]



EPOCH 1/80
  Metric              Train        Val          Δ
  Loss               0.1845     0.1817    -0.0028
  Recall             0.6422     0.8694    +0.2271
  Precision          0.4076     0.4410    +0.0334
  Dice               0.4987     0.5852    +0.0865
  Specificity        0.9987     0.9990    +0.0003
  NPV                0.9995     0.9999    +0.0004
  MCC                0.5108     0.6188    +0.1080
  LR: 1.00e-04 → 9.89e-05
  Dynamic pos_weight (avg): 30.01
  Importance mean  (dec4→dec1): 0.499 | 0.500 | 0.501 | 0.502
  Importance max   (dec4→dec1): 0.503 | 0.505 | 0.508 | 0.509
  GPU: 1.5GB alloc / 18.6GB reserved
  ✓ Best recall: 0.8694  (dice=0.5852  precision=0.4410  MCC=0.6188)
  No-improve streak: 0/30


...

Epoch 5/80 [Val]  : 100%|██████████| 1720/1720 [06:34<00:00,  4.37it/s]



EPOCH 5/80
  Metric              Train        Val          Δ
  Loss               0.0859     0.1446    +0.0587
  Recall             0.8365     0.8539    +0.0174
  Precision          0.7230     0.6979    -0.0251
  Dice               0.7756     0.7680    -0.0075
  Specificity        0.9996     0.9997    +0.0001
  NPV                0.9998     0.9999    +0.0001
  MCC                0.7773     0.7717    -0.0056
  LR: 8.36e-05 → 7.52e-05
  Dynamic pos_weight (avg): 30.00
  Importance mean  (dec4→dec1): 0.497 | 0.499 | 0.501 | 0.503
  Importance max   (dec4→dec1): 0.506 | 0.508 | 0.510 | 0.516
  GPU: 1.5GB alloc / 18.6GB reserved
  No-improve streak: 3/30


Epoch 6/80 [Val]  : 100%|██████████| 1720/1720 [06:33<00:00,  4.37it/s]



EPOCH 6/80
  Metric              Train        Val          Δ
  Loss               0.0811     0.1441    +0.0630
  Recall             0.8498     0.8761    +0.0263
  Precision          0.7380     0.6815    -0.0566
  Dice               0.7900     0.7666    -0.0234
  Specificity        0.9996     0.9996    +0.0000
  NPV                0.9998     0.9999    +0.0001
  MCC                0.7916     0.7724    -0.0192
  LR: 7.52e-05 → 6.58e-05
  Dynamic pos_weight (avg): 30.00
  Importance mean  (dec4→dec1): 0.497 | 0.499 | 0.501 | 0.504
  Importance max   (dec4→dec1): 0.506 | 0.508 | 0.510 | 0.517
  GPU: 1.5GB alloc / 18.6GB reserved
  No-improve streak: 4/30


Epoch 7/80 [Train] lr=6.58e-05:  14%|█▍        | 1780/12380 [14:54<1:29:02,  1.98it/s, loss=0.0625]